In [ ]:
using Plots
using FFTW
using LinearAlgebra


# Elementary Fourier Examples

## Example 1

In [ ]:
N = 64
x = LinRange(0, 2π, N+1)[1:end-1];
@show h = x[2] - x[1];
f = cos.(x);

plot(x, f,marker=:o)

In [ ]:
f

In [ ]:
fhat = fft(f);
round.(fhat, digits=3)

## Example 2

In [ ]:
N = 16
x = LinRange(0, 2π, N+1)[1:end-1];
@show h = x[2] - x[1];
f = sin.(x);
fhat = fft(f);
round.(fhat, digits=3)

## Example 3

In [ ]:
N = 16
x = LinRange(0, 2π, N+1)[1:end-1];
f = @. 1 + sin(x) + cos(3x);
fhat = fft(f);
round.(fhat, digits=3)

# Inverse Transform

In [ ]:
N = 64;
x = LinRange(0, 2π, N+1)[1:end-1];
fhat = zeros(N);
fhat[5+1] = 3/2 * N;
fhat[end-(5-1)] = 3/2 * N;
f = ifft(fhat);
ftrue = 3 * cos.(5*x);
@show norm(f - ftrue, Inf);
plot(x, ftrue, label="Truth")
scatter!(x, real.(f), marker=:o, label="Reconstruction")
xlabel!("x")

Regardless of the input, `fft` and `ifft` return `Complex` type arrays.  Even if the result is real valued (up to floating point)

# Spectarl Differentiation

## Example 1

In [ ]:
N = 8;
x = LinRange(0, 2π, N+1)[1:end-1];
@show h = x[2]-x[1];
f = @. cos(x);
fhat = fft(f);
k = [0:N÷2; -N÷2+1:-1];
@show k;
dfhat = im * k .* fhat;
df = ifft(dfhat);
dftrue = -sin.(x);
@show norm(df - dftrue, Inf);
plot(x, dftrue, label="Truth")
scatter!(x, real.(df), marker=:o, label="Reconstruction")
xlabel!("x")

Compare with centered finite differences

In [ ]:
df_df = (f[3:end] - f[1:end-2]) / (2*h); # this computes at the interior points, x1, x2,..., xN-1
plot(x, dftrue, label="Truth")
scatter!(x, real.(df), marker=:o, label="Reconstruction")
scatter!(x[2:end-1], df_df, marker=:x, label="Finite Difference")
xlabel!("x")

In [ ]:
4/2

## Convergence of Spectral Differentiation

In [ ]:
f_true = x-> exp(sin(x));
df_true = x-> exp(sin(x)) * cos(x);
N_vals = [8, 16, 32, 64, 128, 256, 512, 1024];
errors = zeros(length(N_vals));
for (i, N) in enumerate(N_vals)
    x = LinRange(0, 2π, N+1)[1:end-1];
    h = x[2] - x[1];
    f = f_true.(x);
    fhat = fft(f);
    k = [0:N÷2; -N÷2+1:-1];
    dfhat = im * k .* fhat;
    df = ifft(dfhat);
    dftrue = df_true.(x);
    errors[i] = norm(df - dftrue, Inf);
end
plot(N_vals, errors, marker=:o, xscale=:log2, yscale=:log10, xlabel="N", ylabel="Infinity Norm of Error", title="Error vs N",label="")
yticks!(10.0 .^(-16:2:0))
xticks!(2.0.^(3:10))

# Power Sepctrum

# Example 1

In [ ]:
N = 64;
x = LinRange(0, 2π, N+1)[1:end-1];
f = @. cos(x);
fhat = fft(f);
k = [0:N÷2; -N÷2+1:-1];
scatter(k, abs.(fhat)/N, marker=:o, label="Magnitude of (Scaled) Fourier Coefficients")
xlabel!("k")
ylabel!("|f̂ₖ|/N, Power Spectrum")

## Example 2

In [ ]:
N = 16;
x = LinRange(0, 2π, N+1)[1:end-1];
f = @. exp(sin(x));
fhat = fft(f);
k = [0:N÷2; -N÷2+1:-1];
scatter(k, abs.(fhat)/N, marker=:o, label="",yscale=:log10)
xlabel!("k")
ylabel!("|f̂ₖ|/N, Power Spectrum")

For smooth, $2\pi$ periodic functions, expect faster than polynomial decay of the Fourier coefficients, i.e., something like:
$$
|\hat{f}_k/N|\propto e^{- c k}
$$

# Fourier with a nonperiodic function

In [ ]:
N = 128;
x = LinRange(0, 2π, N+1)[1:end-1];
f = x;
fhat = fft(f);
k = [0:N÷2; -N÷2+1:-1];
scatter(k, abs.(fhat)/N, marker=:o, label="", yscale=:log10)
xlabel!("k")
ylabel!("|f̂ₖ|/N, Power Spectrum")

1. This interprets $f(x) =x$ as its $2\pi$ periodic extension, which is not smooth
2. Nonsmooth functions have **very** broad Fourier support.
   
Fourier methods are not great and/or need to be used with great care, in the nonsmooth setting.